In [54]:
#! pip install prometheus-api-client pandas matplotlib

In [55]:
import pandas as pd
from prometheus_api_client import PrometheusConnect
import matplotlib.pyplot as plt

# 1. Connect to Prometheus
# Use 'http://localhost:9090' if running locally
# Use 'http://prometheus:9090' if inside the same Docker network
prom = PrometheusConnect(url="http://localhost:9090", disable_ssl=True)


+ In Prometheus, the time duration in brackets—the range vector selector—defines the look-back window that the rate() function uses to calculate the per-second average. So, for `rate(messages_consumed_total[**60m**])`, the **60m** indicates the following:
    - The rate() function calculates how much the counter `(total messages)` increased between the start and the end of that **60-minute** window, then divides that increase by the number of seconds in an hour ($3,600$ seconds).

## Metrics per partition

+ **Avg Msg Sent/per sec (60m)**: Measures the average rate at which the producer is publishing new messages to Kafka over the last hour.

+ **Avg Msg Consumed/per sec (60m) = System Throughput**: Tracks the average throughput of the consumers, showing how many messages are being successfully processed per second.

+ **Total Total Msg consumed**: Tracks the total number of messages consumed given a partition.

+ **Consumer Lag (Total)**: The numerical gap between total messages produced and total messages consumed. A rising number here indicates that the consumers are not keeping up with the producer, which is the primary signal that you need to spin up more consumer instances.

+ **Active Consumers**: Displays the current count of unique consumer instances connected to the system, helping verify if our consumer scaling is working as expected.

+ **P50 Latency (sec)**: Indicates the 50th percentile of end-to-end processing time; it means 50% of messages were processed within this duration, highlighting the "average" performance.

+ **P95 Latency (sec)**: Indicates the 95th percentile of end-to-end processing time; it means 95% of messages were processed within this duration, highlighting the "worst-case" performance for most users.

+ **P99 Latency**: The "outlier" experience; essential for finding extreme edge cases.

In [73]:
import pandas as pd

# 1. Get the list of Partitions dynamically
partition_info = prom.custom_query(query='count(count by (partition) (messages_produced_total))')
num_partitions = int(partition_info[0]['value'][1]) if partition_info else 0
partition_list = list(range(num_partitions))

# 2. Define Separate Query Sets
# These keys must match so we can pair them in the loop
partition_queries = {
    'Avg Msg Sent/per sec (10m)': 'sum(rate(messages_produced_total[10m])) by (partition)',
    'Avg Msg Consumed/per sec (10m)': 'sum(rate(messages_consumed_total[10m])) by (partition)',
    'Total Msg consumed': 'sum(messages_consumed_total) by (partition)',
    'Consumer Lag (Total)': 'sum(messages_produced_total) by (partition) - sum(messages_consumed_total) by (partition)',
    'Active Consumers': 'count(messages_consumed_total) by (partition)',
    'P50 Latency last 60m (sec)': 'histogram_quantile(0.50, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le, partition))',
    'P95 Latency last 60m (sec)': 'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le, partition))',
    'P99 Latency last 60m (sec)': 'histogram_quantile(0.99, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le, partition))'
}

all_queries = {
    'Avg Msg Sent/per sec (10m)': 'sum(rate(messages_produced_total[10m]))',
    'Avg Msg Consumed/per sec (10m)': 'sum(rate(messages_consumed_total[10m]))',
    'Total Msg consumed': 'sum(messages_consumed_total)',
    'Consumer Lag (Total)': 'sum(messages_produced_total) - sum(messages_consumed_total)',
    'Active Consumers': 'count(count by (instance) (messages_consumed_total))',
    'P50 Latency last 60m (sec)': 'histogram_quantile(0.50, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le))',
    'P95 Latency last 60m (sec)': 'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le))',
    'P99 Latency last 60m (sec)': 'histogram_quantile(0.99, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le))'
}

rows = []

# 3. Fetch Data
for label in all_queries.keys():
    metric_row = {'Metric': label}
    
    try:
        # A. Fetch the 'all' value using the specific global query
        res_all = prom.custom_query(query=all_queries[label])
        metric_row['All'] = round(float(res_all[0]['value'][1]), 5) if res_all else 0.0
        
        # B. Fetch the partition-specific values
        res_part = prom.custom_query(query=partition_queries[label])
        partition_map = {
            res['metric'].get('partition', 'unknown'): round(float(res['value'][1]), 5)
            for res in res_part
        }
        
        # C. Map individual partitions to columns
        for p in partition_list:
            val = partition_map.get(str(p), 0.0)
            # Logic: If the metric is Active Consumers, cap the partition value at 1
            if label == 'Active Consumers' and val > 1:
                val = 1
            metric_row[f"P_{p}"] = val     
                   
    except Exception as e:
        print(f"Error fetching {label}: {e}")
        metric_row['All'] = "Error"
        for p in partition_list:
            metric_row[f"P_{p}"] = "Error"
            
    rows.append(metric_row)

# 4. Create and Display
summary_df = pd.DataFrame(rows)

print(f"\n=== System Overview: {num_partitions} Partitions Detected ===")

# Apply rounding and formatting
def format_values(val):
    if isinstance(val, (int, float)):
        # If it's a whole number, show as int, else show 3 decimals
        return f"{int(val)}" if val == int(val) else f"{val:.3f}"
    return val

display(summary_df.style.format(format_values, subset=summary_df.columns.drop('Metric')).hide(axis='index'))
# display(summary_df.style.hide(axis='index').set_properties(**{'text-align': 'center'}))


=== System Overview: 14 Partitions Detected ===


Metric,All,P_0,P_1,P_2,P_3,P_4,P_5,P_6,P_7,P_8,P_9,P_10,P_11,P_12,P_13
Avg Msg Sent/per sec (10m),3.142,0,0.030,0.061,0.091,0.121,0.151,0.182,0.212,0.242,0.272,0.289,0.333,0.363,0.371
Avg Msg Consumed/per sec (10m),4.294,0,0.029,0.058,0.087,0.118,0.144,0.182,0.212,0.242,0.269,0.442,0.328,0.970,0.774
Total Msg consumed,13503,0,147,294,441,588,735,882,1029,1176,1323,1327,1176,1532,1485
Consumer Lag (Total),1939,0,0,0,0,0,5,0,0,0,0,145,441,232,426
Active Consumers,9,0,1,1,1,1,1,1,1,1,1,1,1,1,1
P50 Latency last 60m (sec),650.971,0,2.886,3.995,4.042,4.051,201.867,112.289,130.181,132.687,102.736,770.931,800,800,800
P95 Latency last 60m (sec),800,0,696.754,800,800,717.235,800,746.479,800,800,720.334,800,800,800,800
P99 Latency last 60m (sec),800,0,796.193,800,800,800,800,800,800,800,784.067,800,800,800,800


## Metrics per consumer

In [88]:
import pandas as pd

# 1. Get the list of Unique Consumers dynamically
consumer_info = prom.custom_query(query='count by (consumer_id) (messages_consumed_total)')
consumer_list = [res['metric'].get('consumer_id') for res in consumer_info]
num_consumers = len(consumer_list)

# 2. Fetch the Partition Mapping
mapping_query = 'sum by (consumer_id, partition) (rate(messages_consumed_total[15m]) > 0)'
mapping_res = prom.custom_query(query=mapping_query)

assignment_map = {}
for res in mapping_res:
    cid = res['metric'].get('consumer_id')
    part = res['metric'].get('partition')
    if cid not in assignment_map:
        assignment_map[cid] = []
    assignment_map[cid].append(part)

# 3. Define Metric Queries
consumer_queries = {
    'Avg Msg Consumed/per sec (10m)': 'sum(rate(messages_consumed_total[10m])) by (consumer_id)',
    'Msgs Consumed (Last 5m)': 'sum(increase(messages_consumed_total[5m])) by (consumer_id)',
    'Total Msg consumed': 'sum(messages_consumed_total) by (consumer_id)',
    'P50 Latency last 60m (sec)': 'histogram_quantile(0.50, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le, consumer_id))',
    'P95 Latency last 60m (sec)': 'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le, consumer_id))',
    'P99 Latency last 60m (sec)': 'histogram_quantile(0.99, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le, consumer_id))'
}

all_queries = {
    'Avg Msg Consumed/per sec (10m)': 'sum(rate(messages_consumed_total[10m]))',
    'Msgs Consumed (Last 5m)': 'sum(increase(messages_consumed_total[5m]))',
    'Total Msg consumed': 'sum(messages_consumed_total)',
    'P50 Latency last 60m (sec)': 'histogram_quantile(0.50, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le))',
    'P95 Latency last 60m (sec)': 'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le))',
    'P99 Latency last 60m (sec)': 'histogram_quantile(0.99, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le))'
}

#4. Fetch and Build Rows
rows = []

# A. Assigned Partitions Row
partition_row = {'Metric': 'Assigned Partitions', 'All Consumers': 'All'}
for cid in consumer_list:
    display_name = cid if len(cid) < 12 else f"...{cid[-6:]}"
    parts = assignment_map.get(cid, [])
    partition_row[display_name] = ", ".join(sorted(parts, key=int)) if parts else "None"
rows.append(partition_row)

# B. Numerical Metric Rows
for label in all_queries.keys():
    metric_row = {'Metric': label}
    try:
        res_all = prom.custom_query(query=all_queries[label])
        metric_row['All Consumers'] = round(float(res_all[0]['value'][1]), 5) if res_all else 0.0
        
        res_cons = prom.custom_query(query=consumer_queries[label])
        consumer_map = {
            res['metric'].get('consumer_id', 'unknown'): round(float(res['value'][1]), 5)
            for res in res_cons
        }
        
        for cid in consumer_list:
            display_name = cid if len(cid) < 12 else f"...{cid[-6:]}"
            metric_row[display_name] = consumer_map.get(cid, 0.0)
            
    except Exception as e:
        print(f"Error fetching {label}: {e}")
        metric_row['All Consumers'] = "Error"
    rows.append(metric_row)

# 5. Format and Display
summary_df = pd.DataFrame(rows)

def format_values(val):
    if isinstance(val, (int, float)):
        # Display as integer if no decimal part (ideal for the 5m count)
        return f"{int(val)}" if val == int(val) else f"{val:.3f}"
    return val

display(summary_df.style.format(format_values, subset=summary_df.columns.drop('Metric')).hide(axis='index'))

Metric,All Consumers,...c83550,...13eb58,...f436be,...8d40c0,...3f458e,...c3a4b2,...6fe550,...dc58a1,...73bfe4
Assigned Partitions,All,"1, 2, 3, 4, 6, 7","5, 6, 7, 8, 9","10, 11, 12, 13, 14","9, 12","11, 13",1,"10, 11","2, 3","4, 5"
Avg Msg Consumed/per sec (10m),6.002,0.373,0.711,1.614,0.959,1.038,0.019,1.032,0.093,0.163
Msgs Consumed (Last 5m),2390.103,118.996,169.848,497.337,465.812,504.458,9.153,494.324,47.801,82.382
Total Msg consumed,15154,5092,3944,4118,584,629,12,611,60,104
P50 Latency last 60m (sec),662.209,348.691,76.737,800,572.028,800,0.092,800,0.525,1.596
P95 Latency last 60m (sec),800,800,800,800,800,800,0.914,800,3.778,4.660
P99 Latency last 60m (sec),800,800,800,800,800,800,0.994,800,4.756,4.932


## Latency graphs

In [58]:
### Roy add latency graphs per consumer
# add latency graphs per partition
# add overall latency graph


### Latency by partition

In [90]:
import plotly.express as px
from datetime import datetime, timedelta
minutes_interval = 10.0
time_bucket_for_average = 5 #minutes

# 2. Setup Time Range (Last 12 hours)
end_time = datetime.now()
start_time = end_time - timedelta(hours=12)
step = "1m"

query = f'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[{time_bucket_for_average}m])) by (le, partition))'

# 3. Fetch Data
result = prom.custom_query_range(
    query=query,
    start_time=start_time,
    end_time=end_time,
    step=step,
)

# 4. Flatten the result into a DataFrame
df_list = []
for series in result:
    partition = series['metric'].get('partition', 'all')
    for val in series['values']:
        df_list.append({
            'timestamp': pd.to_datetime(val[0], unit='s'),
            'partition': f"Partition {partition}",
            'latency_sec': float(val[1])
        })

df = pd.DataFrame(df_list)
df['timestamp'] = pd.to_datetime(df['timestamp'])

# --- NEW: Calculate the "All" line ---
# Group by timestamp to get the average P95 across all partitions at each point in time
df_all = df.groupby('timestamp')['latency_sec'].mean().reset_index()
df_all['partition'] = 'All Partitions'

# Combine back into the main dataframe
df_all = pd.concat([df, df_all], ignore_index=True)
df_all['timestamp'] = pd.to_datetime(df_all['timestamp'])


######## order the partitions properly (Partition 1, Partition 2, ..., All Partitions)
partitions = [p for p in df_all['partition'].unique() if p != 'All Partitions']

# 2. Sort them based on the digits inside the string
partitions.sort(key=lambda x: int(''.join(filter(str.isdigit, x))))

# 3. Put "All Partitions" at the top (or bottom) of the list
ordered_partitions = ['All Partitions'] + partitions

# 4. Apply this order to the DataFrame
df_all['partition'] = pd.Categorical(df_all['partition'], categories=ordered_partitions, ordered=True)


# 5. Create Plotly Graph
fig = px.line(
    df_all, 
    x='timestamp', 
    y='latency_sec', 
    color='partition',
    category_orders={'partition': ordered_partitions}, # Ensure the legend follows the desired order
    title='P95 End-to-End Latency (Interactive)',
    labels={'timestamp': 'Time', 'latency_sec': 'Percentile 95th Latency (Seconds)'},
    template='plotly_white'
)

# Convert to milliseconds for Plotly
ms_interval = minutes_interval * 60 * 1000
# Logic to determine the best format
if minutes_interval < 60:
    # Under 1 hour: Show minutes and seconds
    custom_format = "%H:%M:%S"
elif minutes_interval < 1440: # 24 hours
    # 1 to 24 hours: Show Hour:Minute and Short Date
    custom_format = "%H:%M\n%b %d"
else:
    # Over 24 hours: Focus on the Date
    custom_format = "%b %d\n%Y"

fig.update_xaxes(
    type='date',             # Force the axis to recognize dates
    tickmode='linear',
    tick0=df['timestamp'].min(), 
    dtick=ms_interval,
    tickformat=custom_format,
    ticks="outside",
    ticklen=10,
    tickcolor='black',
    showgrid=True,
    gridcolor='LightPink'
)
# Choose which one should stay visible
# visible_partition = "Partition 1"
visible_partition = "All Partitions"

fig.for_each_trace(lambda trace: trace.update(visible=True if trace.name == visible_partition else "legendonly"))

fig.show()

# Display Stats Table
# --- Calculate Time-Windowed Stats ---
# Define our window boundaries
now = df['timestamp'].max()  # Using the max timestamp in data as 'now'
ten_min_ago = now - pd.Timedelta(minutes=10)
thirty_min_ago = now - pd.Timedelta(minutes=30)

# Create windowed slices
df_10m = df[df['timestamp'] >= ten_min_ago]
df_30m = df[df['timestamp'] >= thirty_min_ago]

# 1. Aggregate main stats
stats_table = df.groupby('partition').agg(
    mean_latency_sec=('latency_sec', 'mean'),
    max_latency_sec=('latency_sec', 'max'),
    std_latency_sec=('latency_sec', 'std')
).reset_index()

# 2. Join the windowed means
stats_10m = df_10m.groupby('partition')['latency_sec'].mean().rename('p95_last_10m')
stats_30m = df_30m.groupby('partition')['latency_sec'].mean().rename('p95_last_30m')

stats_table = stats_table.merge(stats_10m, on='partition', how='left')
stats_table = stats_table.merge(stats_30m, on='partition', how='left')

# 3. Handle Sorting
stats_table['partition_num'] = stats_table['partition'].str.extract('(\d+)').astype(int)
stats_table = stats_table.sort_values('partition_num').drop(columns=['partition_num']).reset_index(drop=True)

# 4. Calculate the "All Partitions" summary row including windows
summary_row = pd.DataFrame({
    'partition': ['All Partitions'],
    'mean_latency_sec': [df['latency_sec'].mean()],
    'max_latency_sec': [df['latency_sec'].max()],
    'std_latency_sec': [df['latency_sec'].std()],
    'p95_last_10m': [df_10m['latency_sec'].mean()],
    'p95_last_30m': [df_30m['latency_sec'].mean()]
})

# Combine
stats_table = pd.concat([stats_table, summary_row], ignore_index=True)

# 5. Formatting for display
print("### Latency Overview")
print("These statistics summarize the 95th percentile (P95) processing times.")
print("The 'Mean' represents the average P95 over the duration, while 'Max' captures the highest latency spike.")
print("-" * 30)

display(stats_table.style.format({
    'mean_latency_sec': '{:.3f}s',
    'max_latency_sec': '{:.3f}s',
    'std_latency_sec': '{:.3f}s',
    'p95_last_10m': '{:.3f}s',
    'p95_last_30m': '{:.3f}s'
}).set_caption("Performance Metrics: Full Window vs. Recent Windows"))

### Latency Overview
These statistics summarize the 95th percentile (P95) processing times.
The 'Mean' represents the average P95 over the duration, while 'Max' captures the highest latency spike.
------------------------------


,partition,mean_latency_sec,max_latency_sec,std_latency_sec,p95_last_10m,p95_last_30m
0,Partition 1,199.594s,800.000s,285.362s,0.866s,2.088s
1,Partition 2,256.670s,800.000s,340.212s,2.666s,2.009s
2,Partition 3,235.295s,800.000s,316.437s,4.455s,4.425s
3,Partition 4,199.351s,800.000s,285.301s,4.522s,4.542s
4,Partition 5,285.211s,800.000s,320.084s,4.657s,5.460s
5,Partition 6,255.026s,800.000s,291.393s,5.704s,5.974s
6,Partition 7,260.743s,800.000s,290.368s,8.818s,8.675s
7,Partition 8,274.444s,800.000s,326.466s,5.432s,5.054s
8,Partition 9,263.420s,780.000s,308.679s,7.053s,6.834s
9,Partition 10,602.839s,800.000s,262.937s,515.229s,674.701s


## Latency by Consumers

In [91]:
import plotly.express as px
from datetime import datetime, timedelta
minutes_interval = 10.0
time_bucket_for_average = 5 #minutes

# 2. Setup Time Range (Last 12 hours)
end_time = datetime.now()
start_time = end_time - timedelta(hours=12)
step = "1m"

query = f'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[{time_bucket_for_average}m])) by (le, consumer_id))'

# 3. Fetch Data
result = prom.custom_query_range(
    query=query,
    start_time=start_time,
    end_time=end_time,
    step=step,
)

# 4. Flatten the result into a DataFrame
df_list = []
for series in result:
    consumer_id = series['metric'].get('consumer_id', 'all')
    for val in series['values']:
        df_list.append({
            'timestamp': pd.to_datetime(val[0], unit='s'),
            'consumer_id': f"Consumer {consumer_id}",
            'latency_sec': float(val[1])
        })

df = pd.DataFrame(df_list)
df['timestamp'] = pd.to_datetime(df['timestamp'])

# --- NEW: Calculate the "All" line ---
# Group by timestamp to get the average P95 across all consumers at each point in time
df_all = df.groupby('timestamp')['latency_sec'].mean().reset_index()
df_all['consumer_id'] = 'All Consumers'

# Combine back into the main dataframe
df_all = pd.concat([df, df_all], ignore_index=True)
df_all['timestamp'] = pd.to_datetime(df_all['timestamp'])


######## order the consumers properly (Consumer 1, Consumer 2, ..., All Consumers)
consumers = [c for c in df_all['consumer_id'].unique() if c != 'All Consumers']

# 2. Sort them based on the digits inside the string
consumers.sort(key=lambda x: int(''.join(filter(str.isdigit, x))))

# 3. Put "All Consumers" at the top (or bottom) of the list
ordered_consumers = ['All Consumers'] + consumers

# 4. Apply this order to the DataFrame
df_all['consumer_id'] = pd.Categorical(df_all['consumer_id'], categories=ordered_consumers, ordered=True)


# 5. Create Plotly Graph
fig = px.line(
    df_all, 
    x='timestamp', 
    y='latency_sec', 
    color='consumer_id',
    category_orders={'consumer_id': ordered_consumers}, # Ensure the legend follows the desired order
    title='P95 End-to-End Latency (Interactive)',
    labels={'timestamp': 'Time', 'latency_sec': 'Percentile 95th Latency (Seconds)'},
    template='plotly_white'
)

# Convert to milliseconds for Plotly
ms_interval = minutes_interval * 60 * 1000
# Logic to determine the best format
if minutes_interval < 60:
    # Under 1 hour: Show minutes and seconds
    custom_format = "%H:%M:%S"
elif minutes_interval < 1440: # 24 hours
    # 1 to 24 hours: Show Hour:Minute and Short Date
    custom_format = "%H:%M\n%b %d"
else:
    # Over 24 hours: Focus on the Date
    custom_format = "%b %d\n%Y"

fig.update_xaxes(
    type='date',             # Force the axis to recognize dates
    tickmode='linear',
    tick0=df['timestamp'].min(), 
    dtick=ms_interval,
    tickformat=custom_format,
    ticks="outside",
    ticklen=10,
    tickcolor='black',
    showgrid=True,
    gridcolor='LightPink'
)
# Choose which one should stay visible
# visible_consumer = "Consumer 1"
visible_consumer = "All Consumers"

fig.for_each_trace(lambda trace: trace.update(visible=True if trace.name == visible_consumer else "legendonly"))

fig.show()


#######################################################################################
# Query for total messages processed per consumer
# count_query = f'sum(increase(message_processing_latency_seconds_count[{time_bucket_for_average}m])) by (consumer_id)'
# count_query = f'sum(increase(messages_consumed_total[1m])) by (consumer_id)'
count_query = 'sum(messages_consumed_total) by (consumer_id)'
count_result = prom.custom_query_range(
    query=count_query,
    start_time=start_time,
    end_time=end_time,
    step=step,
)

# Flatten count data
count_list = []
for series in count_result:
    cid = series['metric'].get('consumer_id', 'all')
    total_count = sum(float(val[1]) for val in series['values']) # Sum of all increases in the range
    count_list.append({'consumer_id': f"Consumer {cid}", 'msg_count': total_count})

df_counts = pd.DataFrame(count_list)
#######################################################################################

# Display Stats Table
# --- Calculate Time-Windowed Stats ---
# We use the most recent timestamp in the data as our reference point
latest_time = df['timestamp'].max()
ten_min_ago = latest_time - timedelta(minutes=10)
thirty_min_ago = latest_time - timedelta(minutes=30)

# Create subsets for the windows
df_10m = df[df['timestamp'] >= ten_min_ago]
df_30m = df[df['timestamp'] >= thirty_min_ago]

# 1. Aggregate main stats (Mean, Max, Std)
stats_table = df.groupby('consumer_id').agg(
    mean_latency_sec=('latency_sec', 'mean'),
    max_latency_sec=('latency_sec', 'max'),
    std_latency_sec=('latency_sec', 'std')
).reset_index()

# 2. Calculate windowed means per consumer
win_10m = df_10m.groupby('consumer_id')['latency_sec'].mean().rename('p95_10m_sec')
win_30m = df_30m.groupby('consumer_id')['latency_sec'].mean().rename('p95_30m_sec')

# 3. Merge windows into the main table
stats_table = stats_table.merge(win_10m, on='consumer_id', how='left')
stats_table = stats_table.merge(win_30m, on='consumer_id', how='left')

## 4. Merge message counts and calculate percentage
stats_table = stats_table.merge(df_counts, on='consumer_id', how='left').fillna(0)
total_msgs_global = stats_table['msg_count'].sum()
stats_table['pct_of_total_msgs'] = (stats_table['msg_count'] / total_msgs_global) * 100


# 5. Numeric sorting for consumers
#stats_table['consumer_num'] = stats_table['consumer_id'].str.extract('(\d+)').astype(int)
stats_table = stats_table.sort_values('pct_of_total_msgs', ascending=False).drop(columns=['msg_count']).reset_index(drop=True)

# 6. Calculate "All Consumers" summary row including windows
summary_row = pd.DataFrame({
    'consumer_id': ['All Consumers'],
    'mean_latency_sec': [df['latency_sec'].mean()],
    'max_latency_sec': [df['latency_sec'].max()],
    'std_latency_sec': [df['latency_sec'].std()],
    'p95_10m_sec': [df_10m['latency_sec'].mean()],
    'p95_30m_sec': [df_30m['latency_sec'].mean()],
    'pct_of_total_msgs': [100.0] # Sum of percentages is 100
})

# Combine
stats_table = pd.concat([stats_table, summary_row], ignore_index=True)

# --- Display ---
print("### Latency Overview")
print("These statistics summarize the 95th percentile (P95) processing times.")
print(f"Windowed stats calculated relative to last data point: {latest_time.strftime('%H:%M:%S')}")
print("-" * 120)

display(stats_table.style.format({
    'mean_latency_sec': '{:.3f}s',
    'max_latency_sec': '{:.3f}s',
    'std_latency_sec': '{:.3f}s',
    'p95_10m_sec': '{:.3f}s',
    'p95_30m_sec': '{:.3f}s',
    'pct_of_total_msgs': '{:.1f}%'
}))

### Latency Overview
These statistics summarize the 95th percentile (P95) processing times.
Windowed stats calculated relative to last data point: 14:55:47
------------------------------------------------------------------------------------------------------------------------


,consumer_id,mean_latency_sec,max_latency_sec,std_latency_sec,p95_10m_sec,p95_30m_sec,pct_of_total_msgs
0,Consumer 8a8ee4e1c72c-c83550,307.370s,800.000s,347.537s,7.965s,6.194s,50.2%
1,Consumer 8f0cf331470c-13eb58,260.491s,800.000s,347.730s,6.181s,6.493s,24.3%
2,Consumer da02a08fa270-f436be,768.606s,800.000s,121.013s,662.684s,751.275s,19.8%
3,Consumer 8306c8ab396d-6fe550,641.120s,800.000s,264.507s,554.459s,641.120s,2.0%
4,Consumer 0ed9b598f258-3f458e,543.178s,800.000s,350.747s,403.094s,543.178s,1.7%
5,Consumer e1564a74787b-8d40c0,438.557s,800.000s,379.392s,241.406s,438.557s,1.4%
6,Consumer 06afe9fbed83-73bfe4,4.623s,4.680s,0.028s,4.612s,4.623s,0.4%
7,Consumer 17b8365f7142-dc58a1,3.991s,4.400s,0.411s,4.207s,3.991s,0.2%
8,Consumer 659169498b34-c3a4b2,0.866s,0.961s,0.118s,0.845s,0.866s,0.0%
9,All Consumers,366.536s,800.000s,372.163s,209.495s,267.828s,100.0%
